# CEM4644 - MP4: Segmentation for a quantity take-off

## Workshop (in class): *Three 1940 USDA farmhouse plans*

**No coding needed.** Each grey box below is one *step*: click the (play) button at its left, wait until it finishes,
look at the result, then answer the report question that follows. Run the steps **from top to bottom**.

**What you will do (about 90 minutes)**
1. Meet SAM 3 on an ordinary site photo: ask by name, draw a box, tap an object. Then look at the three drawings.
2. Ask for rooms, doors and windows **by name** on a drawing, and see what the drawing's own answer key says.
3. Do the **take-off**: set the scale, box every window, measure every room in square feet - one cell per drawing.
4. Look at where the model goes wrong: the words, the weak regions, and words of your own.
5. Try a drawing of your own.

**Before you start:** menu *Runtime -> Change runtime type -> T4 GPU -> Save*. The model used here (SAM 3) is large:
with a GPU each request takes well under a second; without one the steps that need the model take about a minute each.

Everything is in **feet and square feet**. There is no scale printed on a drawing that you can trust blindly: you set
the scale yourself, from a dimension the drawing prints or from something whose real size you know.

In [ ]:
#@title ▶ Step 0 · Run me first (2-3 minutes) { display-mode: "form" }
#@markdown Click the play button and wait for the 'Ready' line. This downloads the drawings with their answer keys and loads SAM 3 (about 3 GB).
#@markdown Untick *load_model* only if you have no GPU and want to skip the steps that need the live model.
load_model = True #@param {type:"boolean"}
import importlib, os, shutil, subprocess, sys
REPO, FOLDER, PKG = "CEM4644", "mp4_segmentation", "aec_seg"

def _git(*args):
    return subprocess.run(["git", "-C", REPO, *args], capture_output=True, text=True).returncode == 0

if os.path.isdir(REPO):                      # a copy is already here: pull the newest course code over it
    if not (_git("fetch", "-q", "--depth", "1", "origin", "master")
            and _git("reset", "-q", "--hard", "FETCH_HEAD") and _git("clean", "-qfd")):
        shutil.rmtree(REPO, ignore_errors=True)          # broken copy: start again from scratch
if not os.path.isdir(REPO):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "https://github.com/Haolan-Zhang/CEM4644.git", REPO], check=True)
for _m in [m for m in list(sys.modules) if m == PKG or m.startswith(PKG + ".")]:
    del sys.modules[_m]                      # Python caches imported code: drop it, or this cell keeps the old version
importlib.invalidate_caches()
sys.path.insert(0, os.path.abspath(os.path.join(REPO, FOLDER)))
from aec_seg import lab
lab.setup(dataset="workshop", load_model=load_model)


## Part 1 · Meet SAM 3

Detection (MP3) draws a **box** around an object. **Segmentation** goes one step further: it decides, *pixel by pixel*,
what belongs to the object. Count the pixels and you have an area; know the scale and you have square feet. That is what
makes it useful for a **quantity take-off**.

The model is **SAM 3** (Segment Anything Model 3, Meta 2025). You do not train it, and it has no fixed list of classes.
You tell it *what* or *where*, in one of three ways:

- a **phrase**, such as *helmet* or *wet concrete*: it returns every region that matches, each with a **confidence**;
- a **box** around one object: it cuts out that object's exact outline;
- a **click** on one object: the same thing, from a single point.

Two steps on an ordinary site photo first, so that you see what the model does before it meets a drawing.

In [ ]:
#@title ▶ Step 1a · Ask by name { display-mode: "form" }
#@markdown Pick a phrase, or type your own in *own_phrase* (it wins when it is not empty). Three panels: the photo, the mask (white = the model says *this is it*), the overlay. Try a thing (*helmet*), a material (*wet concrete*), a part (*hand*), and something that is not in the photo at all. Watch the confidences.
photo = "pour: Workers pouring and levelling concrete on a slab" #@param ["pour: Workers pouring and levelling concrete on a slab", "mixer: A mixer truck delivering concrete next to a brick house"]
phrase = "person" #@param ["person", "helmet", "safety vest", "boots", "hose", "rebar", "wet concrete", "hand", "truck", "wheel", "brick wall", "sky", "window"]
own_phrase = "" #@param {type:"string"}
confidence = 0.3 #@param {type:"slider", min:0.1, max:0.9, step:0.05}
lab.intro_phrase(photo, phrase, own_phrase, confidence)


In [ ]:
#@title ▶ Step 1b · Box it, or tap it { display-mode: "form" }
#@markdown Draw a box around an object and label it *box*; or draw a tiny box on an object and label it *point* (its centre is the click). Draw several, click *Submit*: SAM 3 cuts out one object per box or click, no words needed. Needs the live model.
photo = "pour: Workers pouring and levelling concrete on a slab" #@param ["pour: Workers pouring and levelling concrete on a slab", "mixer: A mixer truck delivering concrete next to a brick house"]
lab.intro_draw(photo)


### The drawings

Three small farmhouse floor plans published by the U.S. Department of Agriculture in 1940 (public domain). Black walls, drawn windows and door swings, a printed size inside most rooms and overall dimension lines along two sides. There is no scale bar on any of them: you set the scale yourself from a printed dimension. Each drawing comes with an answer key (every room's drawn area, every window, every door) that the notebook checks your measurements against.

Read a drawing before you measure it: the room names and the printed room sizes, the overall dimension lines along two
sides (that is where your scale comes from), the black walls, the windows drawn as a gap with thin lines in it, and the
quarter-circle arcs that are door swings.

In [ ]:
#@title ▶ Step 1c · Browse the drawings { display-mode: "form" }
#@markdown *all drawings* shows all three with what is on them and what you will take off from each. These facts come from the answer key the notebook checks your measurements against.
drawing = "all drawings" #@param ["all drawings", "usda_5544: Five-room farmhouse, 27 ft x 34 ft (USDA design 710-5544)", "usda_5540: Four-room and attic farmhouse, 36 ft x 28 ft (USDA design 710-5540)", "usda_5539: Four-room farmhouse, 38 ft x 28 ft (USDA design 710-5539)"]
lab.show_sheets(drawing)


## Part 2 · Ask for something by name

Pick a drawing and a thing. You get three panels: the drawing, the **mask** (white = the model says *this is it*), and
the **overlay**. Below them: how many regions, how many pixels, how many square feet, and what the **drawing's own
answer key** says. The **confidence slider** hides the regions the model is unsure about: watch the count change.

In [ ]:
#@title ▶ Step 2a · Original, mask, overlay { display-mode: "form" }
#@markdown Try *room (any)* first, then *bedroom*, then *kitchen*, *porch*, *door*, *window*. Some words work, some find nothing at all: that is the lesson of this part.
drawing = "usda_5544: Five-room farmhouse, 27 ft x 34 ft (USDA design 710-5544)" #@param ["usda_5544: Five-room farmhouse, 27 ft x 34 ft (USDA design 710-5544)", "usda_5540: Four-room and attic farmhouse, 36 ft x 28 ft (USDA design 710-5540)", "usda_5539: Four-room farmhouse, 38 ft x 28 ft (USDA design 710-5539)"]
thing = "room (any)" #@param ["room (any)", "bedroom", "kitchen", "living room", "bathroom", "porch", "closet", "door", "window", "wall", "stairs", "fireplace"]
confidence = 0.3 #@param {type:"slider", min:0.1, max:0.9, step:0.05}
lab.segment(drawing, thing, confidence)


In [ ]:
#@title ▶ Step 2b · Hits, misses and extras { display-mode: "form" }
#@markdown The answer key drawn on the drawing: green = a real one the model found, red = a real one it missed, blue = a region that is not one at all.
drawing = "usda_5544: Five-room farmhouse, 27 ft x 34 ft (USDA design 710-5544)" #@param ["usda_5544: Five-room farmhouse, 27 ft x 34 ft (USDA design 710-5544)", "usda_5540: Four-room and attic farmhouse, 36 ft x 28 ft (USDA design 710-5540)", "usda_5539: Four-room farmhouse, 38 ft x 28 ft (USDA design 710-5539)"]
thing = "room (any)" #@param ["room (any)", "bedroom", "kitchen", "living room", "bathroom", "porch", "closet", "door", "window", "wall", "stairs", "fireplace"]
confidence = 0.3 #@param {type:"slider", min:0.1, max:0.9, step:0.05}
lab.count(drawing, thing, confidence)


> ### 📝 Report question 1
> From Steps 2a and 2b: which words found what they should (rooms? bedrooms? doors? windows?), and which found nothing or something else? Give the found / missed / extra numbers for two things on one drawing at confidence 0.3, and say what the misses have in common.

## Part 3 · The take-off

A phrase is quick, but a take-off needs control, so now **you** draw the boxes. One cell per drawing. In each cell, pick
the label above the picture before you draw, and draw in this order:

1. **the scale**: one box exactly along a printed overall dimension, from arrowhead to arrowhead. Only the length along
   that dimension is used. *A 1 % error in the scale is a 2 % error in every area, because area is scale squared.*
2. **every window**: a small box on each. There are no round numbers to guess here - the notebook checks your boxes
   against the drawing.
3. **every room**: a tight box each, the edges on the **inside faces** of the walls.

Then press *Submit*. You can also ask the model to **find everything that looks like your first box** of something
(the dropdown under the picture) - that is how repeated symbols get counted on a big sheet.

How a *room* box becomes square feet: a box on its own makes SAM 3 cut out the *furniture symbols* inside it rather than
the floor (it was trained to find objects). So the notebook asks for *empty room* **and** hands it your box, keeps the
region that fits your box, fills the holes the symbols leave, removes the black walls, gives back the bites that door
swings take out of a rectangular room, and converts the pixels with **your** scale.

In [ ]:
#@title ▶ Step 3a · Take-off: usda_5544, Five-room farmhouse, 27 ft x 34 ft (USDA design 710-5544) { display-mode: "form" }
#@markdown **Five-room farmhouse, 27 ft x 34 ft (USDA design 710-5544).** Set the scale: box the 27'-0" dimension at the top from arrowhead tip to arrowhead tip (label: scale). Box every window (9 of them; find-all does not work on this sheet). Box every door, arc and all (17 of them). Box every room, porch, hall and closet and compare your areas with the sizes printed on the sheet.
#@markdown Zoom with the mouse wheel. Pick the label above the picture before each box. Rooms need the live model. Copy the tables into your report.
lab.takeoff("usda_5544: Five-room farmhouse, 27 ft x 34 ft (USDA design 710-5544)")


In [ ]:
#@title ▶ Step 3b · Take-off: usda_5540, Four-room and attic farmhouse, 36 ft x 28 ft (USDA design 710-5540) { display-mode: "form" }
#@markdown **Four-room and attic farmhouse, 36 ft x 28 ft (USDA design 710-5540).** Set the scale: box the 36'-0" dimension at the top from arrowhead tip to arrowhead tip (label: scale). Box every window (11 of them), then try 'find all like my first box' at 0.3 on the whole sheet. Box every door, arc and all (14 of them). Box every room, porch, hall and closet; watch what the fireplace chimney does to the PORCH.
#@markdown Zoom with the mouse wheel. Pick the label above the picture before each box. Rooms need the live model. Copy the tables into your report.
lab.takeoff("usda_5540: Four-room and attic farmhouse, 36 ft x 28 ft (USDA design 710-5540)")


In [ ]:
#@title ▶ Step 3c · Take-off: usda_5539, Four-room farmhouse, 38 ft x 28 ft (USDA design 710-5539) { display-mode: "form" }
#@markdown **Four-room farmhouse, 38 ft x 28 ft (USDA design 710-5539).** Set the scale: box the 38'-0" dimension from arrowhead tip to arrowhead tip (label: scale). Box every window (10 of them), then try 'find all like my first box' at 0.3 on the whole sheet. Box every door, arc and all (11 of them). Box every room, porch, hall and closet. This plan is L-shaped: the kitchen runs on south into an un-lettered dining corner, and the fireplace takes a bite out of the living room.
#@markdown Zoom with the mouse wheel. Pick the label above the picture before each box. Rooms need the live model. Copy the tables into your report.
lab.takeoff("usda_5539: Four-room farmhouse, 38 ft x 28 ft (USDA design 710-5539)")


> ### 📝 Report question 2
> From Step 3 on all three drawings: your scale reading on each drawing and how far it is from the answer key; your window count (found / missed / extra); and the room table (your square feet, the drawing's, the error) for the drawing you did best on. What is the total of your rooms against the drawing's indoor total?

> ### 📝 Report question 3
> Which rooms came out worst, and why? Look at the pictures and name the reason for at least three of them (a loose box, a kitchen counter or a bathtub eaten out of the mask, a hall that is really a set of doorways, an open space with no wall on one side, the scale). If you used the *find all like my first box* dropdown, say what it found and what it missed.

## Part 4 · Where it goes wrong

Three kinds of error to look for: the **words** you use (the model was trained on everyday photographs, not on drawings),
the **weak regions** it proposes with a low confidence, and words of your own that describe what is *drawn* rather than
what it *means*.

In [ ]:
#@title ▶ Step 4a · Does the wording matter? { display-mode: "form" }
#@markdown The same thing asked for with four different words. All the wordings are precomputed, so this is instant. Try *door* (against *curved line*) and *wall* (against *thick black line*).
drawing = "usda_5544: Five-room farmhouse, 27 ft x 34 ft (USDA design 710-5544)" #@param ["usda_5544: Five-room farmhouse, 27 ft x 34 ft (USDA design 710-5544)", "usda_5540: Four-room and attic farmhouse, 36 ft x 28 ft (USDA design 710-5540)", "usda_5539: Four-room farmhouse, 38 ft x 28 ft (USDA design 710-5539)"]
thing = "door" #@param ["room (any)", "bedroom", "kitchen", "living room", "bathroom", "porch", "closet", "door", "window", "wall", "stairs", "fireplace"]
confidence = 0.3 #@param {type:"slider", min:0.1, max:0.9, step:0.05}
lab.phrase_lab(drawing, thing, confidence)


In [ ]:
#@title ▶ Step 4b · Look at each region and its confidence { display-mode: "form" }
#@markdown Every region the model proposed, numbered, with its confidence, its area and the room it sits on. Move the slider to see which ones survive.
drawing = "usda_5544: Five-room farmhouse, 27 ft x 34 ft (USDA design 710-5544)" #@param ["usda_5544: Five-room farmhouse, 27 ft x 34 ft (USDA design 710-5544)", "usda_5540: Four-room and attic farmhouse, 36 ft x 28 ft (USDA design 710-5540)", "usda_5539: Four-room farmhouse, 38 ft x 28 ft (USDA design 710-5539)"]
thing = "room (any)" #@param ["room (any)", "bedroom", "kitchen", "living room", "bathroom", "porch", "closet", "door", "window", "wall", "stairs", "fireplace"]
lab.inspect(drawing, thing)


In [ ]:
#@title ▶ Step 4c · Your own words { display-mode: "form" }
#@markdown Type any phrase: a room, a symbol, a shape. Try *curved line* (the door swings), *thick black line* (the walls), *circle*, *small rectangle*, *hatched square*. Needs the live model.
drawing = "usda_5544: Five-room farmhouse, 27 ft x 34 ft (USDA design 710-5544)" #@param ["usda_5544: Five-room farmhouse, 27 ft x 34 ft (USDA design 710-5544)", "usda_5540: Four-room and attic farmhouse, 36 ft x 28 ft (USDA design 710-5540)", "usda_5539: Four-room farmhouse, 38 ft x 28 ft (USDA design 710-5539)"]
phrase = "curved line" #@param {type:"string"}
confidence = 0.3 #@param {type:"slider", min:0.1, max:0.9, step:0.05}
lab.your_phrase(drawing, phrase, confidence)


> ### 📝 Report question 4
> From Step 4a: which wording worked best for the thing you chose, and how different were the counts? From Step 4b: describe one weak region (what it sits on, its confidence) and one plain mistake. What would you tell a colleague who wants to put these square feet in a cost estimate?

> ### 📝 Report question 5
> From Step 4c: which of your own words found something that the name of the thing could not (for example *curved line* for the door swings, *thick black line* for the walls)? Why does a shape word work on a drawing where the name of the thing does not?

## Part 5 · Your own drawing

In [ ]:
#@title ▶ Your drawing, your words { display-mode: "form" }
#@markdown This cell prints a **link**: open it in a new tab (it works on a phone too). Upload a floor plan (a photograph of a drawing works), type what to find, move the threshold.
#@markdown To get square feet, measure a printed dimension on your own sheet first: count the pixels along it, divide by its length in feet, and type that number in. Test at least one drawing of your own and take screenshots. Needs the live model.
lab.upload_app()


> ### 📝 Report question 6
> Test one drawing of your own (any floor plan or construction drawing). What phrase did you use, what did it find, and was the mask right? Then: where in a project would a take-off like this be useful, and where would it mislead you? What would you need (clean drawings, a known dimension, a room schedule, a person checking) before you would put these numbers in an estimate?

## Wrap-up

In [ ]:
#@title ▶ Numbers for your report { display-mode: "form" }
#@markdown Every take-off you submitted in Part 3, printed again in one place.
lab.report_summary()


### Where the drawings come from, and the model

- **usda_5544** - USDA design 710-5544, 'Five-room farmhouse', Miscellaneous Publication 360 'Plans of Farm Buildings for Southern States' (1940), p. 17. U.S. Department of Agriculture, Bureau of Agricultural Engineering. Public domain (work of the U.S. Government, 17 U.S.C. 105). https://archive.org/details/plansoffarmbuild360unit
- **usda_5540** - USDA design 710-5540, 'Four-room and attic farmhouse', Miscellaneous Publication 360 'Plans of Farm Buildings for Southern States' (1940), p. 13. U.S. Department of Agriculture, Bureau of Agricultural Engineering. Public domain (work of the U.S. Government, 17 U.S.C. 105). https://archive.org/details/plansoffarmbuild360unit
- **usda_5539** - USDA design 710-5539, 'Four-room farmhouse', Miscellaneous Publication 360 'Plans of Farm Buildings for Southern States' (1940), p. 12. U.S. Department of Agriculture, Bureau of Agricultural Engineering. Public domain (work of the U.S. Government, 17 U.S.C. 105). https://archive.org/details/plansoffarmbuild360unit
- Site photos in Part 1: Workers pouring and levelling concrete on a slab (U.S. Air Force / Airman Sydney Franklin, Public domain, https://upload.wikimedia.org/wikipedia/commons/0/0e/Concrete_pouring_for_the_new_Spangdahlem_Elementary_School_%288062807%29.jpg); A mixer truck delivering concrete next to a brick house (Kolforn, CC BY-SA 4.0, https://upload.wikimedia.org/wikipedia/commons/5/57/-2021-01-18_Foundations_and_concrete_oversite%2C_Trimingham%2C_Norfolk_%283%29.JPG).
- Model: SAM 3 by Meta AI (SAM License), loaded from a public mirror of the official checkpoint; a copy of the licence is in `docs/SAM_LICENSE.txt`.
- Lab code: https://github.com/Haolan-Zhang/CEM4644 (folder `mp4_segmentation`).